In [1]:
# =========================
# IMPORTAR LIBRERÍAS
# =========================

import pandas as pd
import numpy as np

In [3]:
# =========================
# CARGAR DATASET FINAL Y TABLAS DEL EDA
# =========================

df = pd.read_csv("../data/processed/dataset_final_aire_meteo_zaragoza_limpio.csv")
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce", format="mixed")

resumen_contaminante = pd.read_csv("../data/processed/resumen_contaminante.csv")
tabla_estacion_contaminante = pd.read_csv("../data/processed/tabla_estacion_contaminante.csv")
tabla_correlaciones = pd.read_csv("../data/processed/tabla_correlaciones_meteorologicas.csv")
tabla_resumen_final = pd.read_csv("../data/processed/tabla_resumen_final.csv")

print("Dataset principal:", df.shape)
print("Resumen contaminante:", resumen_contaminante.shape)
print("Tabla estación-contaminante:", tabla_estacion_contaminante.shape)
print("Tabla correlaciones:", tabla_correlaciones.shape)
print("Tabla resumen final:", tabla_resumen_final.shape)

C:\Users\kevin\AppData\Local\Temp\ipykernel_29240\3327850594.py:5: DtypeWarning: Columns (0: datetime, 1: estacion_meteo, 2: provincia_meteo, 3: prec_traza) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/dataset_final_aire_meteo_zaragoza_limpio.csv")


Dataset principal: (1304856, 43)
Resumen contaminante: (7, 8)
Tabla estación-contaminante: (44, 8)
Tabla correlaciones: (7, 7)
Tabla resumen final: (28, 7)


In [4]:
# =========================
# RESUMEN GENERAL DEL PROYECTO
# =========================
# Aquí dejamos métricas globales que luego servirán
# para redactar conclusiones y también para el README.

resumen_general = {
    "registros_totales": len(df),
    "contaminantes_analizados": df["contaminante"].nunique(),
    "estaciones_analizadas": df["estacion"].nunique(),
    "fecha_inicio": df["datetime"].min(),
    "fecha_fin": df["datetime"].max(),
    "registros_sin_nulos_valor": df["valor"].notna().sum()
}

resumen_general

{'registros_totales': 1304856,
 'contaminantes_analizados': 7,
 'estaciones_analizadas': 8,
 'fecha_inicio': Timestamp('2020-01-01 00:00:00'),
 'fecha_fin': Timestamp('2023-11-30 00:00:00'),
 'registros_sin_nulos_valor': np.int64(1260016)}

In [5]:
# =========================
# CONTAMINANTES CON MAYOR CONCENTRACIÓN MEDIA
# =========================
# Ordenamos de mayor a menor media para detectar
# qué contaminantes destacan más en el análisis.

top_media_contaminantes = (
    resumen_contaminante
    .sort_values("media", ascending=False)
    .reset_index(drop=True)
)

display(top_media_contaminantes)

,contaminante,unidad,registros,media,mediana,desviacion_std,minimo,maximo
0,O3,µg/m3,273226,50.935175,53.47,27.758891,0.10,173.12
1,NO2,µg/m3,270369,20.023261,15.86,14.021811,1.00,177.73
2,PM10,µg/m3,186286,18.194829,15.15,13.634353,0.00,168.30
3,PM2.5,µg/m3,19615,8.571669,7.45,6.192152,0.10,93.69
4,NO,µg/m3,66351,7.760430,4.68,11.307598,0.00,327.71
5,SO2,µg/m3,206463,4.306574,3.99,1.898421,0.03,42.48
6,CO,mg/m3,237706,0.198016,0.18,0.087498,0.02,2.19


In [6]:
# =========================
# ESTACIONES CON MAYOR CONCENTRACIÓN MEDIA
# =========================
# Tomamos la tabla estación-contaminante y vemos
# qué estaciones destacan más en promedio.

top_estaciones = (
    tabla_estacion_contaminante
    .groupby("estacion_label", as_index=False)
    .agg(media_global=("media", "mean"))
    .sort_values("media_global", ascending=False)
    .reset_index(drop=True)
)

display(top_estaciones)

,estacion_label,media_global
0,Estación 40,23.080891
1,Estación 39,17.836856
2,Estación 29,17.587080
3,Estación 36,17.094101
4,Estación 38,17.059047
5,Estación 26,16.960429
6,Estación 37,16.512590
7,Estación 32,16.197081


In [9]:
# =========================
# CONCLUSIONES METEOROLÓGICAS
# =========================
# Mostramos la tabla de correlaciones para ver
# qué variables meteorológicas parecen relacionarse más
# con cada contaminante.

display(tabla_correlaciones)

correlaciones_ordenadas = tabla_correlaciones.copy()

# Creamos columnas con valor absoluto para detectar
# relaciones más intensas, aunque sean negativas.
for col in [
    "corr_temp_media",
    "corr_precipitacion",
    "corr_viento",
    "corr_presion_max",
    "corr_presion_min"
]:
    correlaciones_ordenadas[f"{col}_abs"] = correlaciones_ordenadas[col].abs()

display(correlaciones_ordenadas)

,contaminante,unidad,corr_temp_media,corr_precipitacion,corr_viento,corr_presion_max,corr_presion_min
0,CO,mg/m3,-0.187620,0.044124,-0.273089,0.142309,0.140852
1,NO,µg/m3,-0.263926,-0.044956,-0.237177,0.283865,0.269706
2,NO2,µg/m3,-0.197319,0.012571,-0.318566,0.187525,0.176268
3,O3,µg/m3,0.448462,-0.055160,0.300905,-0.234565,-0.219388
4,PM10,µg/m3,0.095766,-0.035822,-0.255135,0.119894,0.126421
5,PM2.5,µg/m3,0.052845,-0.003186,-0.360518,0.194382,0.198149
6,SO2,µg/m3,-0.153893,-0.065508,-0.140084,0.135241,0.113609


,contaminante,unidad,corr_temp_media,corr_precipitacion,corr_viento,corr_presion_max,corr_presion_min,corr_temp_media_abs,corr_precipitacion_abs,corr_viento_abs,corr_presion_max_abs,corr_presion_min_abs
0,CO,mg/m3,-0.187620,0.044124,-0.273089,0.142309,0.140852,0.187620,0.044124,0.273089,0.142309,0.140852
1,NO,µg/m3,-0.263926,-0.044956,-0.237177,0.283865,0.269706,0.263926,0.044956,0.237177,0.283865,0.269706
2,NO2,µg/m3,-0.197319,0.012571,-0.318566,0.187525,0.176268,0.197319,0.012571,0.318566,0.187525,0.176268
3,O3,µg/m3,0.448462,-0.055160,0.300905,-0.234565,-0.219388,0.448462,0.055160,0.300905,0.234565,0.219388
4,PM10,µg/m3,0.095766,-0.035822,-0.255135,0.119894,0.126421,0.095766,0.035822,0.255135,0.119894,0.126421
5,PM2.5,µg/m3,0.052845,-0.003186,-0.360518,0.194382,0.198149,0.052845,0.003186,0.360518,0.194382,0.198149
6,SO2,µg/m3,-0.153893,-0.065508,-0.140084,0.135241,0.113609,0.153893,0.065508,0.140084,0.135241,0.113609


In [10]:
# =========================
# PATRONES TEMPORALES
# =========================
# Resumimos la concentración media, mediana y máxima
# por contaminante y estación del año.

patrones_temporales = (
    df.groupby(["contaminante", "estacion_anio"], as_index=False)
    .agg(
        media=("valor", "mean"),
        mediana=("valor", "median"),
        maximo=("valor", "max"),
        registros=("valor", "size")
    )
    .sort_values(["contaminante", "media"], ascending=[True, False])
)

display(patrones_temporales.head(30))

,contaminante,estacion_anio,media,mediana,maximo,registros
0,CO,Invierno,0.231098,0.21,1.89,60168
1,CO,Otoño,0.197684,0.18,1.35,61152
3,CO,Verano,0.182344,0.17,0.92,61824
2,CO,Primavera,0.181419,0.17,2.19,61824
4,NO,Invierno,12.382936,7.34,248.79,16560
5,NO,Otoño,8.929112,5.20,327.71,17472
6,NO,Primavera,5.520186,4.04,129.27,17664
7,NO,Verano,4.437058,3.51,122.42,17664
8,NO2,Invierno,25.492950,21.73,177.73,68592
9,NO2,Otoño,21.878877,17.69,165.68,69888


In [11]:
# =========================
# LIMITACIONES DEL ANÁLISIS
# =========================
# Dejamos por escrito las limitaciones principales
# para reutilizarlas luego en el README o en el informe.

limitaciones = [
    "Los datos meteorológicos proceden de una estación de referencia y se han unido por fecha con los datos de calidad del aire.",
    "El análisis se ha centrado en Zaragoza y en el periodo 2020-2023.",
    "Existen valores nulos en algunas mediciones de contaminación, lo que puede afectar ligeramente a algunos resúmenes.",
    "Las correlaciones meteorológicas no implican causalidad directa, solo asociación estadística.",
    "Las unidades de los contaminantes no son homogéneas, por lo que no conviene comparar directamente todos los contaminantes con el mismo eje."
]

for i, texto in enumerate(limitaciones, start=1):
    print(f"{i}. {texto}")

1. Los datos meteorológicos proceden de una estación de referencia y se han unido por fecha con los datos de calidad del aire.
2. El análisis se ha centrado en Zaragoza y en el periodo 2020-2023.
3. Existen valores nulos en algunas mediciones de contaminación, lo que puede afectar ligeramente a algunos resúmenes.
4. Las correlaciones meteorológicas no implican causalidad directa, solo asociación estadística.
5. Las unidades de los contaminantes no son homogéneas, por lo que no conviene comparar directamente todos los contaminantes con el mismo eje.


In [12]:
# =========================
# TEXTO BASE DE CONCLUSIONES DEL PROYECTO
# =========================
# Generamos un texto que luego podrás reutilizar
# en el README o en el informe final.

contaminante_top = top_media_contaminantes.loc[0, "contaminante"]
unidad_top = top_media_contaminantes.loc[0, "unidad"]
media_top = round(top_media_contaminantes.loc[0, "media"], 2)

estacion_top = top_estaciones.loc[0, "estacion_label"]
media_estacion_top = round(top_estaciones.loc[0, "media_global"], 2)

texto_conclusiones = f"""
Conclusiones del análisis

El análisis de la calidad del aire en Zaragoza entre 2020 y 2023 ha permitido estudiar
el comportamiento de varios contaminantes atmosféricos y su relación con variables meteorológicas.

Entre los principales hallazgos, destaca que el contaminante con mayor concentración media
ha sido {contaminante_top}, con una media aproximada de {media_top} {unidad_top}.

A nivel espacial, la estación con mayor concentración media global ha sido {estacion_top},
con una media aproximada de {media_estacion_top} considerando el conjunto de contaminantes analizados.

El estudio también ha permitido identificar patrones temporales, diferencias entre estaciones
y relaciones entre contaminación y meteorología. Estas relaciones deben interpretarse como asociaciones
estadísticas y no como relaciones causales directas.

En conjunto, el proyecto permite construir un dashboard operativo útil para analizar la evolución
de la calidad del aire en Zaragoza, comparar estaciones, estudiar contaminantes concretos
y apoyar la interpretación de los datos mediante variables meteorológicas.
"""

print(texto_conclusiones)


Conclusiones del análisis

El análisis de la calidad del aire en Zaragoza entre 2020 y 2023 ha permitido estudiar
el comportamiento de varios contaminantes atmosféricos y su relación con variables meteorológicas.

Entre los principales hallazgos, destaca que el contaminante con mayor concentración media
ha sido O3, con una media aproximada de 50.94 µg/m3.

A nivel espacial, la estación con mayor concentración media global ha sido Estación 40,
con una media aproximada de 23.08 considerando el conjunto de contaminantes analizados.

El estudio también ha permitido identificar patrones temporales, diferencias entre estaciones
y relaciones entre contaminación y meteorología. Estas relaciones deben interpretarse como asociaciones
estadísticas y no como relaciones causales directas.

En conjunto, el proyecto permite construir un dashboard operativo útil para analizar la evolución
de la calidad del aire en Zaragoza, comparar estaciones, estudiar contaminantes concretos
y apoyar la interpretac

In [13]:
# =========================
# EXPORTACIÓN FINAL DE APOYO AL INFORME
# =========================
# Guardamos tablas y un archivo de texto con las conclusiones.

# Guardar patrones temporales
patrones_temporales.to_csv("../data/processed/patrones_temporales.csv", index=False)

# Guardar ranking de estaciones
top_estaciones.to_csv("../data/processed/top_estaciones.csv", index=False)

# Guardar texto de conclusiones en txt
with open("../reports/conclusiones_proyecto.txt", "w", encoding="utf-8") as f:
    f.write(texto_conclusiones)

print("Archivos exportados correctamente:")
print("- ../data/processed/patrones_temporales.csv")
print("- ../data/processed/top_estaciones.csv")
print("- ../reports/conclusiones_proyecto.txt")

Archivos exportados correctamente:
- ../data/processed/patrones_temporales.csv
- ../data/processed/top_estaciones.csv
- ../reports/conclusiones_proyecto.txt
